# Sensor placement (MuJoCo) — **medium** Colab run

- **CMA force-reinstall** (below) can fix NumPy / `cma` binary mismatch — **Runtime → Restart** after that cell, then run the rest.
- This notebook **clones** the project and optimizes **five fixed vehicle mounts** (see user config + `configs/mujoco_vehicle.yaml`): **200** generations, checkpoints every 25.

**5 fixed physical slots (only sensor *type* is optimized; no pose / offsets):**
- `top_front_edge_l`, `top_front_edge_r`, `front_face_center`, `side_front_edge_l`, `side_front_edge_r`


In [ ]:
# --- Fix 1 (NumPy / cma ABI): run, then use Runtime → Restart session ---
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "--force-reinstall", "--no-cache-dir", "cma"
])
print("If Colab still complains about NumPy, also run: pip install --no-cache-dir --upgrade numpy")


In [ ]:
# --- Clone + install (after restart, run this cell) ---
import os, subprocess, sys, shutil
from pathlib import Path

WORK = Path("/content")
TARGET = Path("/content/sensor_placement_opt_MUJOCO_ver")
if TARGET.is_dir() and (TARGET / "sensor_opt").is_dir():
    pass
else:
    if TARGET.exists():
        shutil.rmtree(TARGET, ignore_errors=True)
    os.chdir(WORK)
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/dcaglar-28/sensor_placement_opt_MUJOCO_ver.git", str(TARGET.name)])
os.chdir(TARGET)
if str(TARGET) not in sys.path:
    sys.path.insert(0, str(TARGET))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
print("cwd", os.getcwd())


In [ ]:
# --- Imports: verify stack ---
import importlib, sys, os
os.chdir(r"/content/sensor_placement_opt_MUJOCO_ver")
if r"/content/sensor_placement_opt_MUJOCO_ver" not in sys.path:
    sys.path.insert(0, r"/content/sensor_placement_opt_MUJOCO_ver")
import cma, jax, matplotlib, mlflow, mujoco, numpy as np, pandas as pd, rich, scipy, sklearn, torch, yaml
import sensor_opt
print("OK: repo on path, sensor_opt importable; mujoco", mujoco.__version__)

In [ ]:
# ════════════════════════════════════════════════════════════
# USER CONFIGURATION — edit this cell only
# ════════════════════════════════════════════════════════════

# Trial type: "accuracy" | "speed" | "multi_objective"
TRIAL_TYPE = "multi_objective"

# --- Simulation parameters ---
PATH_LENGTH_M      = 20.0   # straight-line path length in meters
VEHICLE_SPEED_MPS  = 2.0    # constant vehicle speed (m/s)
N_OBSTACLES        = 10     # number of obstacles per episode
BASE_RANDOM_SEED   = 42     # seed for obstacle generation

# --- Hardware constraints (enforced as hard limit in multi-objective trial) ---
MAX_HARDWARE_BUDGET_USD = 5000

SENSOR_COSTS_USD = {
    "lidar":    2000,
    "radar":     800,
    "camera":    300,
    "disabled":    0,
}

# Max active sensors (None = no limit, up to 5)
MAX_SENSOR_COUNT = None

# --- Loss weight overrides (None = use defaults in trial_objectives.py) ---
LOSS_WEIGHT_OVERRIDES = None

# ════════════════════════════════════════════════════════════

print(f"Trial          : {TRIAL_TYPE}")
print(f"Path length    : {PATH_LENGTH_M}m at {VEHICLE_SPEED_MPS}m/s")
print(f"Obstacles      : {N_OBSTACLES} per episode (seed={BASE_RANDOM_SEED})")
print(f"Max budget     : ${MAX_HARDWARE_BUDGET_USD:,}")
print(f"Max sensors    : {MAX_SENSOR_COUNT or 'no limit'}")
print("Ready. Run the next cell to start optimization.")


In [ ]:
# --- Full run (takes a while: 200 generations) — uses USER CONFIGURATION cell + mujoco_vehicle.yaml ---
import os, subprocess, sys
from pathlib import Path
import yaml

os.chdir(r"/content/sensor_placement_opt_MUJOCO_ver")
if r"/content/sensor_placement_opt_MUJOCO_ver" not in sys.path:
    sys.path.insert(0, r"/content/sensor_placement_opt_MUJOCO_ver")

from sensor_opt.config.specs import prepare_experiment_config

with open("configs/mujoco_vehicle.yaml") as f:
    raw = yaml.safe_load(f)
rt = {
    "trial_type": TRIAL_TYPE,
    "PATH_LENGTH_M": PATH_LENGTH_M,
    "VEHICLE_SPEED_MPS": VEHICLE_SPEED_MPS,
    "N_OBSTACLES": N_OBSTACLES,
    "BASE_RANDOM_SEED": BASE_RANDOM_SEED,
    "MAX_HARDWARE_BUDGET_USD": MAX_HARDWARE_BUDGET_USD,
    "SENSOR_COSTS_USD": SENSOR_COSTS_USD,
    "MAX_SENSOR_COUNT": MAX_SENSOR_COUNT,
}
ov = {"runtime": rt}
if LOSS_WEIGHT_OVERRIDES is not None:
    ov["loss"] = {"trial_weight_overrides": LOSS_WEIGHT_OVERRIDES}
full = prepare_experiment_config(raw, ov)
outp = Path("configs/_notebook_active.yaml")
outp.write_text(yaml.dump(full, default_flow_style=False, sort_keys=False), encoding="utf-8")
print("Wrote", outp)

subprocess.check_call(
    [sys.executable, "-m", "sensor_opt.run_experiment", "--config", str(outp), "--no-mlflow"]
)
print("Check results/mujoco_*/ and checkpoints/ every 25 gen.")


In [ ]:
# --- Convergence + loss breakdown (uses generations.csv) ---
import os, sys
os.chdir(r"/content/sensor_placement_opt_MUJOCO_ver")
if r"/content/sensor_placement_opt_MUJOCO_ver" not in sys.path:
    sys.path.insert(0, r"/content/sensor_placement_opt_MUJOCO_ver")
from pathlib import Path
import matplotlib.pyplot as plt
from sensor_opt.plotting.cma_matplotlib import plot_cma_generations_matplotlib

cands = sorted(Path("results").glob("mujoco_*/generations.csv"), key=lambda p: p.stat().st_mtime, reverse=True)
assert cands, "No run found — run the experiment cell first"
csv_path = cands[0]
print("Using", csv_path)
fig = plot_cma_generations_matplotlib(csv_path, title=csv_path.parent.name)
plt.show()


In [ ]:
# --- σ vs best loss (dual y) + slot coverage + 2D placement + validation ---
import json, os, sys
os.chdir(r"/content/sensor_placement_opt_MUJOCO_ver")
if r"/content/sensor_placement_opt_MUJOCO_ver" not in sys.path:
    sys.path.insert(0, r"/content/sensor_placement_opt_MUJOCO_ver")
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import yaml
from sensor_opt.plotting.colab_optim_plots import (
    plot_sigma_vs_best_loss, slot_coverage_heatmap, sensor_placement_2d,
)
from sensor_opt.config.specs import prepare_experiment_config

results = sorted(
    Path("results").glob("mujoco_*"),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
assert results, "No results dir — run the experiment cell first."
run_dir = results[0]
print("Run:", run_dir)

with open(run_dir / "config.json") as f:
    full = prepare_experiment_config(yaml.safe_load(f))
mounts = list(full["mounting_slots"])
role_names = [
    "collision (fwd L)",
    "blind / latency (fwd R)",
    "speed (top L)",
    "redundancy (top R)",
    "left lateral",
    "right lateral",
]
ck = sorted((run_dir / "checkpoints").glob("checkpoint_gen*.json"), key=lambda x: x.name)
if ck:
    with open(ck[-1]) as f:
        snap = json.load(f)
    bcfg = snap.get("best_config")
else:
    with open(run_dir / "evaluated_pool.json") as f:
        pool = json.load(f)
    o = [sum(r.get("objectives", {}).values()) for r in pool]
    best = pool[int(np.argmin(o))] if o else {}
    bcfg = best.get("config")

if not bcfg or "sensors" not in bcfg:
    print("No sensor layout found.")
else:
    by_slot = {s["slot"]: s for s in bcfg["sensors"]}
    per = [1.0 if by_slot.get(sl, {}).get("type", "disabled") != "disabled" else 0.0 for sl in mounts]
    print("--- Per-mount coverage (1 = active sensor on mount) ---")
    for m, r, c in zip(mounts, role_names, per):
        t = (by_slot.get(m) or {}).get("type", "n/a")
        print(f"  {m:22s} {r:28s} type={str(t):10s} covered={c:.0f}")
    fig0 = slot_coverage_heatmap(mounts, per, title="Active mount coverage")
    plt.show()
    pts = [(s["slot"], s["x_offset"], s["y_offset"]) for s in bcfg["sensors"] if s.get("type") != "disabled"]
    if pts:
        fig1 = sensor_placement_2d(pts, title="x/y offset (m) per active mount")
        plt.show()

cvp = run_dir / "generations.csv"
if cvp.is_file():
    fig2 = plot_sigma_vs_best_loss(cvp, title=run_dir.name)
    plt.show()
